# Range sensing and occupancy

Turn synthetic range returns into a metric occupancy map.

Run with the repository virtual environment. These are synthetic research experiments; units and assumptions are explicit below.

## Scene and sensor

Returns intersect actor rectangles. No road surface or wall geometry is sensed unless represented as an actor.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from autonomous_vehicles.simulation.actors import static_obstacle
from autonomous_vehicles.simulation.geometry import Pose2, Vec2
from autonomous_vehicles.simulation.occupancy import GridSpec, OccupancyGrid, inflate_obstacles
from autonomous_vehicles.simulation.sensors import RangeSensor, RangeSensorConfig
from autonomous_vehicles.visualization import plot_scan

actors = tuple(
    static_obstacle(f"box-{i}", x, y).at(0) for i, (x, y) in enumerate(((15, 0), (25, 6), (30, -5)))
)
sensor = RangeSensor(RangeSensorConfig(beams=121, noise_std=0.03), seed=42)

## Scan

Non-hit beams are unknown rather than proof of free space.

In [ ]:
scan = sensor.scan(Pose2(), actors)
plot_scan(scan)
plt.show()
print("Hits:", int(scan.hits.sum()), "of", len(scan.hits))
assert scan.ranges.min() >= 0
assert scan.ranges.max() <= sensor.config.maximum_range

## Integrate

Occupied log odds rise at endpoints; free log odds fall along observed beams.

In [ ]:
spec = GridSpec(width=90, height=60, resolution=0.5, origin=Vec2(-5, -15))
grid = OccupancyGrid(spec)
for x in (0, 1, 2):
    grid.integrate(sensor.scan(Pose2(x, 0), actors))
fig, ax = plt.subplots(figsize=(10, 4))
view = ax.imshow(
    grid.probabilities, origin="lower", extent=spec.extent, vmin=0, vmax=1, cmap="gray_r"
)
fig.colorbar(view, ax=ax, label="Occupancy probability")
ax.set_xlabel("x (m)")
ax.set_ylabel("y (m)")
plt.show()

## Unknown cells

Unknown remains 0.5 probability and is tracked by a separate observed mask.

In [ ]:
print("Observed cells:", grid.observed.sum())
print("Unknown cells:", (~grid.observed).sum())
assert np.all(grid.probabilities[~grid.observed] == 0.5)
fig, ax = plt.subplots(figsize=(10, 4))
ax.imshow(grid.observed, origin="lower", extent=spec.extent, cmap="Blues")
ax.set_title("Observed space")
plt.show()

## Inflate for routing

Inflation is a metric-to-grid approximation, not vehicle orientation-aware collision checking.

In [ ]:
occupied = grid.probabilities > 0.65
inflated = inflate_obstacles(occupied, cells=3)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, values, title in zip(
    axes, (occupied, inflated), ("Occupied", "Inflated 1.5 m"), strict=True
):
    ax.imshow(values, origin="lower", extent=spec.extent)
    ax.set_title(title)
    ax.set_xlabel("x (m)")
plt.show()

## Research record

Record the seed, configuration, dependency lock and observed failures with each experiment. Results here demonstrate the implemented model under its stated assumptions. They do not establish performance outside this synthetic distribution.